In [31]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [11]:
!git clone https://github.com/alibabasglab/MossFormer2.git
%cd MossFormer2
!ls
!find . -maxdepth 2 -type d | sort
!find MossFormer2_standalone -maxdepth 2 -type f | sort
!find MossFormer2_samples_Libri2Mix -maxdepth 2 -type f | sort

!pip install -r MossFormer2_standalone/requirements.txt
!cat MossFormer2_standalone/requirements.txt
!head -120 MossFormer2_standalone/inference.py
!head -120 MossFormer2_standalone/model/mossformer2.py
!pip install torchmetrics

Cloning into 'MossFormer2'...
remote: Enumerating objects: 1020, done.
remote: Counting objects: 100% (38/38), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 1020 (delta 9), reused 23 (delta 1), pack-reused 982 (from 1)
Receiving objects: 100% (1020/1020), 60.86 MiB | 34.66 MiB/s, done.
Resolving deltas: 100% (41/41), done.
/kaggle/working/MossFormer2
LICENSE			       MossFormer2_samples_WSJ0-2mix
MossFormer2_samples_Libri2Mix  MossFormer2_samples_WSJ0-3mix
MossFormer2_samples_WHAM       MossFormer2_standalone
MossFormer2_samples_WHAMR      README.md
.
./.git
./.git/branches
./.git/hooks
./.git/info
./.git/logs
./.git/objects
./.git/refs
./MossFormer2_samples_Libri2Mix
./MossFormer2_samples_WHAM
./MossFormer2_samples_WHAMR
./MossFormer2_samples_WSJ0-2mix
./MossFormer2_samples_WSJ0-3mix
./MossFormer2_standalone
./MossFormer2_standalone/model
./MossFormer2_standalone/test_samples
MossFormer2_standalone/inference.py
MossFormer2_standalone/model/__init__.py
MossFormer

In [14]:
# Imports & Configuration
# ==========================================================

import gc
import glob
import os

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchaudio

from tqdm.auto import tqdm

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

from MossFormer2_standalone.model.mossformer2 import Mossformer2Wrapper
from MossFormer2_standalone.model.mossformer2_configs import (
    mossformer2_wsj0mix_3spk,
)
from torchmetrics.audio import ScaleInvariantSignalDistortionRatio

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MAX_SPEAKERS = 8

LATENT_CHANNELS = 512

HIDDEN_DIM = 256

CONFIDENCE_THRESHOLD = 0.35

CROP_SIZE = 16000

torch.backends.cudnn.benchmark = True

sisdr = ScaleInvariantSignalDistortionRatio().to(DEVICE)

print("DEVICE :", DEVICE)

DEVICE : cuda


In [48]:
#Version 1

# ==========================================================
# Load Pretrained Model
# ==========================================================

backbone = Mossformer2Wrapper(
    mossformer2_wsj0mix_3spk
)

backbone.loadPretrained()

backbone.eval()

waveform, sr = torchaudio.load(
    "MossFormer2_samples_WSJ0-3mix/item0_mix.wav"
)

waveform = waveform.to(backbone.device)

with torch.no_grad():

    mix_w = backbone.encoder(waveform)

print("Waveform :", waveform.shape)
print("Encoder  :", mix_w.shape)

# ==========================================================
# Adaptive Multi Scale Temporal Module
# ==========================================================

class AdaptiveMultiScaleTemporalModule(nn.Module):

    def __init__(self, channels=LATENT_CHANNELS):

        super().__init__()

        reduced = channels // 4

        self.reduce = nn.Conv1d(
            channels,
            reduced,
            1,
        )

        self.dw3 = nn.Conv1d(
            reduced,
            reduced,
            3,
            padding=1,
            groups=reduced,
        )

        self.dw5 = nn.Conv1d(
            reduced,
            reduced,
            5,
            padding=2,
            groups=reduced,
        )

        self.dw9 = nn.Conv1d(
            reduced,
            reduced,
            9,
            padding=4,
            groups=reduced,
        )

        self.expand = nn.Conv1d(
            reduced * 3,
            channels,
            1,
        )

        self.refine = nn.Sequential(

            nn.Conv1d(
                channels,
                channels,
                1,
            ),

            nn.GELU()

        )

    def forward(self, x):

        identity = x

        x = self.reduce(x)

        x3 = self.dw3(x)

        x5 = self.dw5(x)

        x9 = self.dw9(x)

        x = torch.cat(
            [x3, x5, x9],
            dim=1,
        )

        x = self.expand(x)

        x = self.refine(x)

        return x + identity

# ==========================================================
# Adaptive Gated Feature Fusion (Stable V2)
# ==========================================================

class AdaptiveGatedFeatureFusion(nn.Module):

    def __init__(self, channels=LATENT_CHANNELS):

        super().__init__()

        self.gate = nn.Sequential(

            nn.Conv1d(
                channels,
                channels,
                kernel_size=1,
                bias=False,
            ),

            nn.BatchNorm1d(channels),

            nn.Sigmoid(),

        )

    def forward(

        self,

        original,

        enhanced,

    ):

        gate = self.gate(original)

        fused = original + 0.1 * gate * enhanced

        return fused
        
# ==========================================================
# Speaker Count Head
# ==========================================================

class SpeakerCountHead(nn.Module):

    def __init__(
        self,
        channels=LATENT_CHANNELS,
        hidden_dim=HIDDEN_DIM,
        max_speakers=MAX_SPEAKERS,
    ):

        super().__init__()

        self.pool = nn.AdaptiveAvgPool1d(1)

        self.classifier = nn.Sequential(

            nn.Linear(channels, hidden_dim),

            nn.GELU(),

            nn.Dropout(0.2),

            nn.Linear(hidden_dim, max_speakers)

        )

    def forward(self, x):

        x = self.pool(x)

        x = x.squeeze(-1)

        return self.classifier(x)

# ==========================================================
# Confidence Head
# ==========================================================

class ConfidenceHead(nn.Module):

    def __init__(
        self,
        channels=LATENT_CHANNELS,
        hidden_dim=HIDDEN_DIM,
        max_speakers=MAX_SPEAKERS,
    ):

        super().__init__()

        self.pool = nn.AdaptiveAvgPool1d(1)

        self.classifier = nn.Sequential(

            nn.Linear(channels, hidden_dim),
        
            nn.GELU(),
        
            nn.Dropout(0.2),
        
            nn.Linear(hidden_dim, max_speakers),
        
        )

    def forward(self, x):

        x = self.pool(x)

        x = x.squeeze(-1)

        return self.classifier(x)

# ==========================================================
# Enhanced Adaptive MossFormer
# ==========================================================

class EnhancedAdaptiveMossFormer(nn.Module):

    def __init__(self, config):

        super().__init__()

        self.backbone = Mossformer2Wrapper(config)

        self.backbone.loadPretrained()

        self.encoder = self.backbone.encoder

        self.masknet = self.backbone.masknet

        self.decoder = self.backbone.decoder

        self.num_spks = self.backbone.num_spks

        self.amtm = AdaptiveMultiScaleTemporalModule()

        self.agff = AdaptiveGatedFeatureFusion()

        self.speaker_counter = SpeakerCountHead()

        self.confidence_head = ConfidenceHead()

        for p in self.encoder.parameters():
            p.requires_grad = False

        for p in self.masknet.parameters():
            p.requires_grad = True

        for p in self.decoder.parameters():
            p.requires_grad = False

        # Train AMTM
        for p in self.amtm.parameters():
            p.requires_grad = True
        
        # Train AGFF
        for p in self.agff.parameters():
            p.requires_grad = True
        
        # Train Speaker Counter
        for p in self.speaker_counter.parameters():
            p.requires_grad = True
        
        # Train Confidence Head
        for p in self.confidence_head.parameters():
            p.requires_grad = True

    @property
    def device(self):
        return next(self.parameters()).device

    def forward(self, mixture):

        mix_w = self.encoder(mixture)

        enhanced = self.amtm(mix_w)

        enhanced = torch.tanh(enhanced)
        
        mix_w = self.agff(
            mix_w,
            enhanced,
        )

        speaker_logits = self.speaker_counter(mix_w)

        confidence = self.confidence_head(mix_w)

        est_mask = self.masknet(mix_w)

        mix_stack = torch.stack(
            [mix_w] * self.num_spks
        )

        sep_h = mix_stack * est_mask

        est_source = torch.cat(
            [
                self.decoder(sep_h[i]).unsqueeze(-1)
                for i in range(self.num_spks)
            ],
            dim=-1,
        )

        T_origin = mixture.size(1)

        T_est = est_source.size(1)

        if T_origin > T_est:

            est_source = F.pad(
                est_source,
                (0,0,0,T_origin-T_est)
            )

        else:

            est_source = est_source[:, :T_origin, :]

        return {

            "audio": est_source,

            "speaker_logits": speaker_logits,

            "confidence": confidence

        }

# ==========================================================
# Verify Model
# ==========================================================

gc.collect()

torch.cuda.empty_cache()

model = EnhancedAdaptiveMossFormer(
    mossformer2_wsj0mix_3spk
).to(DEVICE)

model.eval()

with torch.no_grad():

    outputs = model(waveform)

print("="*60)

print(outputs["audio"].shape)

print(outputs["speaker_logits"].shape)

print(outputs["confidence"].shape)

print("="*60)

total = sum(
    p.numel()
    for p in model.parameters()
)

trainable = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Total :", total)

print("Trainable :", trainable)

# ==========================================================
# Loss Function
# ==========================================================

class EnhancedMossFormerLoss(nn.Module):

    def __init__(
        self,
        separation_weight=0.75,
        speaker_weight=0.15,
        confidence_weight=0.10,
    ):

        super().__init__()

        self.log_sigma_sep = nn.Parameter(torch.zeros(1))
        self.log_sigma_spk = nn.Parameter(torch.zeros(1))
        self.log_sigma_conf = nn.Parameter(torch.zeros(1))

        self.separation_loss = nn.L1Loss()

        self.speaker_loss = nn.CrossEntropyLoss()

        self.confidence_loss = nn.BCEWithLogitsLoss()

    def forward(

        self,

        outputs,

        target_audio,

        target_num_speakers,

        target_confidence,

    ):
        print("audio NaN :", torch.isnan(outputs["audio"]).any().item())
        print("target NaN:", torch.isnan(target_audio).any().item())
        
        print("audio Inf :", torch.isinf(outputs["audio"]).any().item())
        print("target Inf:", torch.isinf(target_audio).any().item())
        
        sep = self.separation_loss(
            outputs["audio"],
            target_audio,
        )
        

        spk = self.speaker_loss(

            outputs["speaker_logits"],

            target_num_speakers,

        )

        conf = self.confidence_loss(

            outputs["confidence"],

            target_confidence,

        )

        weight_sep = torch.exp(
            torch.clamp(-self.log_sigma_sep, -5, 5)
        )
        
        weight_spk = torch.exp(
            torch.clamp(-self.log_sigma_spk, -5, 5)
        )
        
        weight_conf = torch.exp(
            torch.clamp(-self.log_sigma_conf, -5, 5)
        )
        
        loss_sep = (
            weight_sep * sep
            + self.log_sigma_sep
        )
        
        loss_spk = (
            weight_spk * spk
            + self.log_sigma_spk
        )
        
        loss_conf = (
            weight_conf * conf
            + self.log_sigma_conf
        )
        
        total = (
            loss_sep
            + loss_spk
            + loss_conf
        )

        print(
            f"log_sigma_sep  : {self.log_sigma_sep.item():.4f}"
        )
        print(
            f"log_sigma_spk  : {self.log_sigma_spk.item():.4f}"
        )
        print(
            f"log_sigma_conf : {self.log_sigma_conf.item():.4f}"
        )

        return {

            "total_loss": total,

            "separation_loss": sep,

            "speaker_loss": spk,

            "confidence_loss": conf,

            "log_sigma_sep": self.log_sigma_sep.detach(),

            "log_sigma_spk": self.log_sigma_spk.detach(),
            
            "log_sigma_conf": self.log_sigma_conf.detach(),

        }

# ==========================================================
# Optimizer
# ==========================================================

criterion = EnhancedMossFormerLoss().to(DEVICE)
print(next(criterion.parameters()).device)

optimizer = optim.AdamW(

    [
        {
            "params": model.parameters(),
        },
        {
            "params": criterion.parameters(),
        },
    ],

    lr=1e-4,

    weight_decay=1e-5,

)

scheduler = optim.lr_scheduler.CosineAnnealingLR(

    optimizer,

    T_max=20,

)

scaler = torch.amp.GradScaler("cuda")

print("Optimizer Ready")

# ==========================================================
# Dataset
# ==========================================================

class MossFormerDataset(Dataset):

    def __init__(

        self,

        root_dir,

    ):

        self.mixtures = sorted(

            glob.glob(

                os.path.join(

                    root_dir,

                    "*_mix.wav",

                )

            )

        )

    def __len__(self):

        return len(self.mixtures)

    def __getitem__(

        self,

        idx,

    ):

        mix_path = self.mixtures[idx]

        mixture, _ = torchaudio.load(

            mix_path

        )

        src1, _ = torchaudio.load(

            mix_path.replace(

                "_mix.wav",

                "_source1.wav",

            )

        )

        src2, _ = torchaudio.load(

            mix_path.replace(

                "_mix.wav",

                "_source2.wav",

            )

        )

        src3, _ = torchaudio.load(

            mix_path.replace(

                "_mix.wav",

                "_source3.wav",

            )

        )

        length = mixture.shape[1]

        if length > CROP_SIZE:

            start = torch.randint(

                0,

                length - CROP_SIZE,

                (1,),

            ).item()

            end = start + CROP_SIZE

            mixture = mixture[:, start:end]

            src1 = src1[:, start:end]

            src2 = src2[:, start:end]

            src3 = src3[:, start:end]

        sources = torch.stack(

            [

                src1.squeeze(0),

                src2.squeeze(0),

                src3.squeeze(0),

            ],

            dim=-1,

        )

        return {

            "mixture": mixture,

            "sources": sources,

            "num_speakers": torch.tensor(

                3,

                dtype=torch.long,

            ),

            "confidence": torch.tensor(

                [

                    1.,

                    1.,

                    1.,

                    0.,

                    0.,

                    0.,

                    0.,

                    0.,

                ],

                dtype=torch.float32,

            ),

        }

# ==========================================================
# DataLoader
# ==========================================================

def collate_fn(batch):

    mixtures = [

        x["mixture"].squeeze()

        for x in batch

    ]

    mixtures = pad_sequence(

        mixtures,

        batch_first=True,

    )

    sources = [

        x["sources"]

        for x in batch

    ]

    max_len = max(

        s.shape[0]

        for s in sources

    )

    padded = []

    for s in sources:

        if s.shape[0] < max_len:

            s = torch.cat(

                [

                    s,

                    torch.zeros(

                        max_len - s.shape[0],

                        3,

                    ),

                ],

                dim=0,

            )

        padded.append(s)

    return {

        "mixture": mixtures,

        "sources": torch.stack(padded),

        "num_speakers": torch.stack(

            [

                x["num_speakers"]

                for x in batch

            ]

        ),

        "confidence": torch.stack(

            [

                x["confidence"]

                for x in batch

            ]

        ),

    }

train_dataset = MossFormerDataset(

    "MossFormer2_samples_WSJ0-3mix"

)

train_loader = DataLoader(

    train_dataset,

    batch_size=1,

    shuffle=True,

    collate_fn=collate_fn,

)

print(len(train_dataset))
print(len(train_loader))

# ==========================================================
# Train One Epoch
# ==========================================================

def train_one_epoch(

    model,

    dataloader,

    optimizer,

    criterion,

    scaler,

    scheduler=None,

):

    model.train()

    running_loss = 0.0
    running_sep = 0.0
    running_spk = 0.0
    running_conf = 0.0

    progress = tqdm(
        dataloader,
        desc="Training",
    )

    for batch in progress:

        mixture = batch["mixture"].to(
            DEVICE,
            non_blocking=True,
        )

        sources = batch["sources"].to(
            DEVICE,
            non_blocking=True,
        )

        num_speakers = batch["num_speakers"].to(
            DEVICE,
            non_blocking=True,
        )

        confidence = batch["confidence"].to(
            DEVICE,
            non_blocking=True,
        )

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(

            device_type="cuda",

            enabled=False #torch.cuda.is_available(),

        ):

            outputs = model(mixture)

            losses = criterion(

                outputs,

                sources,

                num_speakers,

                confidence,

            )

            if torch.isnan(losses["separation_loss"]):
                print("NaN separation loss detected")
                break
            if torch.isnan(outputs["audio"]).any():
                print("NaN in separated audio")
                
            print("\n========== Forward Check ==========")

            for name, tensor in outputs.items():

                print(

                    name,

                    "NaN:", torch.isnan(tensor).any().item(),

                    "Inf:", torch.isinf(tensor).any().item(),

                )

            print("Total Loss :", losses["total_loss"].item())
            print("Sep Loss   :", losses["separation_loss"].item())
            print("Spk Loss   :", losses["speaker_loss"].item())
            print("Conf Loss  :", losses["confidence_loss"].item())

        scaler.scale(
            losses["total_loss"]
        ).backward()

        print("\n========== Gradient Check ==========")

        for name, param in model.named_parameters():

            if param.grad is None:
                continue

            if torch.isnan(param.grad).any():

                print(f"NaN Gradient : {name}")

                break

            if torch.isinf(param.grad).any():

                print(f"Inf Gradient : {name}")

                break

        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(

            model.parameters(),

            max_norm=1.0,

        )

        scaler.step(optimizer)

        scaler.update()

        for name, param in model.named_parameters():

            if torch.isnan(param).any():

                print(f"NaN weights in {name}")

                raise RuntimeError("NaN weights detected")

        running_loss += losses["total_loss"].item()
        running_sep += losses["separation_loss"].item()
        running_spk += losses["speaker_loss"].item()
        running_conf += losses["confidence_loss"].item()

        progress.set_postfix(

            Loss=f"{running_loss/(progress.n+1):.4f}",

            Sep=f"{running_sep/(progress.n+1):.4f}",

            Spk=f"{running_spk/(progress.n+1):.4f}",

            Conf=f"{running_conf/(progress.n+1):.4f}",

        )

    if scheduler is not None:

        scheduler.step()

    return {

        "loss": running_loss / len(dataloader),

        "separation_loss": running_sep / len(dataloader),

        "speaker_loss": running_spk / len(dataloader),

        "confidence_loss": running_conf / len(dataloader),

    }

# ==========================================================
# Validation
# ==========================================================

@torch.no_grad()
def validate(

    model,

    dataloader,

    criterion,

):

    model.eval()

    running = 0.0
    running_sisdr = 0.0

    for batch in dataloader:

        mixture = batch["mixture"].to(DEVICE)

        target_audio = batch["sources"].to(DEVICE)

        target_num_speakers = batch["num_speakers"].to(DEVICE)

        target_confidence = batch["confidence"].to(DEVICE)

        outputs = model(mixture)

        losses = criterion(

            outputs,

            target_audio,

            target_num_speakers,

            target_confidence,

        )

        score = sisdr(

            outputs["audio"].permute(0, 2, 1),

            target_audio.permute(0, 2, 1),

        )

        running += losses["total_loss"].item()

        running_sisdr += score.item()

    return {

        "loss": running / len(dataloader),

        "sisdr": running_sisdr / len(dataloader),

    }

# ==========================================================
# Train
# ==========================================================

gc.collect()

torch.cuda.empty_cache()

best_loss = float("inf")

NUM_EPOCHS = 20

for epoch in range(NUM_EPOCHS):

    print("\n" + "=" * 60)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
    print("=" * 60)

    history = train_one_epoch(

        model=model,

        dataloader=train_loader,

        optimizer=optimizer,

        criterion=criterion,

        scaler=scaler,

        scheduler=scheduler,

    )

    metrics = validate(

        model,

        train_loader,

        criterion,

    )

    print("\nTraining")

    for key, value in history.items():

        print(f"{key:20}: {value:.5f}")

    print("\nValidation")

    print(f"Loss               : {metrics['loss']:.5f}")

    print(f"SI-SDR             : {metrics['sisdr']:.5f}")

    if metrics["loss"] < best_loss:

        best_loss = metrics["loss"]

        torch.save(

            {

                "model": model.state_dict(),

                "optimizer": optimizer.state_dict(),

                "epoch": epoch,

                "loss": metrics["loss"],

            },

            "best_enhanced_mossformer.pth",

        )

        print("\n✓ Best model saved.")

mossformer2-wsj0mix-3spk config loaded
model initialised on cuda:0
Waveform : torch.Size([1, 44920])
Encoder  : torch.Size([1, 512, 5614])
mossformer2-wsj0mix-3spk config loaded
model initialised on cuda:0
torch.Size([1, 44920, 3])
torch.Size([1, 8])
torch.Size([1, 8])
Total : 57056002
Trainable : 57039618
cuda:0
Optimizer Ready
38
38

Epoch 1/20


Training:   0%|          | 0/38 [00:00<?, ?it/s]

audio NaN : False
target NaN: False
audio Inf : False
target Inf: False
log_sigma_sep  : 0.0000
log_sigma_spk  : 0.0000
log_sigma_conf : 0.0000

========== Forward Check ==========
audio NaN: False Inf: False
speaker_logits NaN: False Inf: False
confidence NaN: False Inf: False
Total Loss : 3.172497510910034
Sep Loss   : 0.3866754472255707
Spk Loss   : 2.097289562225342
Conf Loss  : 0.6885326504707336

========== Gradient Check ==========
audio NaN : False
target NaN: False
audio Inf : False
target Inf: False
log_sigma_sep  : -0.0001
log_sigma_spk  : 0.0001
log_sigma_conf : -0.0001

========== Forward Check ==========
audio NaN: False Inf: False
speaker_logits NaN: False Inf: False
confidence NaN: False Inf: False
Total Loss : 3.1876473426818848
Sep Loss   : 0.41060343384742737
Spk Loss   : 2.0896518230438232
Conf Loss  : 0.6875911951065063

========== Gradient Check ==========
audio NaN : False
target NaN: False
audio Inf : False
target Inf: False
log_sigma_sep  : -0.0002
log_sigma_sp

Training:   0%|          | 0/38 [00:00<?, ?it/s]

audio NaN : False
target NaN: False
audio Inf : False
target Inf: False
log_sigma_sep  : -0.0039
log_sigma_spk  : 0.0036
log_sigma_conf : -0.0039

========== Forward Check ==========
audio NaN: False Inf: False
speaker_logits NaN: False Inf: False
confidence NaN: False Inf: False
Total Loss : 2.3471550941467285
Sep Loss   : 0.11886409670114517
Spk Loss   : 1.618776559829712
Conf Loss  : 0.6167033910751343

========== Gradient Check ==========
audio NaN : False
target NaN: False
audio Inf : False
target Inf: False
log_sigma_sep  : -0.0040
log_sigma_spk  : 0.0037
log_sigma_conf : -0.0040

========== Forward Check ==========
audio NaN: False Inf: False
speaker_logits NaN: False Inf: False
confidence NaN: False Inf: False
Total Loss : 2.3747355937957764
Sep Loss   : 0.09748156368732452
Spk Loss   : 1.6645451784133911
Conf Loss  : 0.6203303337097168

========== Gradient Check ==========
audio NaN : False
target NaN: False
audio Inf : False
target Inf: False
log_sigma_sep  : -0.0042
log_sigm

...

audio NaN : False
target NaN: False
audio Inf : False
target Inf: False
log_sigma_sep  : -0.0418
log_sigma_spk  : -0.0288
log_sigma_conf : -0.0483

========== Forward Check ==========
audio NaN: False Inf: False
speaker_logits NaN: False Inf: False
confidence NaN: False Inf: False
Total Loss : -0.08856409043073654
Sep Loss   : 0.019764656201004982
Spk Loss   : 0.003743665525689721
Conf Loss  : 0.005554738454520702

========== Gradient Check ==========
audio NaN : False
target NaN: False
audio Inf : False
target Inf: False
log_sigma_sep  : -0.0418
log_sigma_spk  : -0.0288
log_sigma_conf : -0.0483

========== Forward Check ==========
audio NaN: False Inf: False
speaker_logits NaN: False Inf: False
confidence NaN: False Inf: False
Total Loss : -0.08664479851722717
Sep Loss   : 0.023539764806628227
Spk Loss   : 0.0032474659383296967
Conf Loss  : 0.004135737195611

========== Gradient Check ==========
audio NaN : False
target NaN: False
audio Inf : False
target Inf: False
log_sigma_sep  : -

Training:   0%|          | 0/38 [00:00<?, ?it/s]

audio NaN : False
target NaN: False
audio Inf : False
target Inf: False
log_sigma_sep  : -0.0420
log_sigma_spk  : -0.0290
log_sigma_conf : -0.0485

========== Forward Check ==========
audio NaN: False Inf: False
speaker_logits NaN: False Inf: False
confidence NaN: False Inf: False
Total Loss : -0.09223480522632599
Sep Loss   : 0.0200180746614933
Spk Loss   : 0.0020881532691419125
Conf Loss  : 0.004041778855025768

========== Gradient Check ==========
audio NaN : False
target NaN: False
audio Inf : False
target Inf: False
log_sigma_sep  : -0.0420
log_sigma_spk  : -0.0290
log_sigma_conf : -0.0485

========== Forward Check ==========
audio NaN: False Inf: False
speaker_logits NaN: False Inf: False
confidence NaN: False Inf: False
Total Loss : -0.09622199833393097
Sep Loss   : 0.010820005089044571
Spk Loss   : 0.005742362700402737
Conf Loss  : 0.005805070511996746

========== Gradient Check ==========
audio NaN : False
target NaN: False
audio Inf : False
target Inf: False
log_sigma_sep  : 

Training:   0%|          | 0/38 [00:00<?, ?it/s]

audio NaN : False
target NaN: False
audio Inf : False
target Inf: False
log_sigma_sep  : -0.0421
log_sigma_spk  : -0.0291
log_sigma_conf : -0.0486

========== Forward Check ==========
audio NaN: False Inf: False
speaker_logits NaN: False Inf: False
confidence NaN: False Inf: False
Total Loss : -0.09435534477233887
Sep Loss   : 0.0209072083234787
Spk Loss   : 0.0009516716236248612
Conf Loss  : 0.0025271624326705933

========== Gradient Check ==========
audio NaN : False
target NaN: False
audio Inf : False
target Inf: False
log_sigma_sep  : -0.0421
log_sigma_spk  : -0.0291
log_sigma_conf : -0.0486

========== Forward Check ==========
audio NaN: False Inf: False
speaker_logits NaN: False Inf: False
confidence NaN: False Inf: False
Total Loss : -0.09320685267448425
Sep Loss   : 0.01881173625588417
Spk Loss   : 0.0017487009754404426
Conf Loss  : 0.004923153668642044

========== Gradient Check ==========
audio NaN : False
target NaN: False
audio Inf : False
target Inf: False
log_sigma_sep  :

In [51]:
#Version 2

# ==========================================================
# Load Pretrained Model
# ==========================================================

backbone = Mossformer2Wrapper(
    mossformer2_wsj0mix_3spk
)

backbone.loadPretrained()

backbone.eval()

waveform, sr = torchaudio.load(
    "MossFormer2_samples_WSJ0-3mix/item0_mix.wav"
)

waveform = waveform.to(backbone.device)

with torch.no_grad():

    mix_w = backbone.encoder(waveform)

print("Waveform :", waveform.shape)
print("Encoder  :", mix_w.shape)

# ==========================================================
# Adaptive Multi Scale Temporal Module
# ==========================================================

class AdaptiveMultiScaleTemporalModule(nn.Module):

    def __init__(self, channels=LATENT_CHANNELS):

        super().__init__()

        reduced = channels // 4

        self.reduce = nn.Conv1d(
            channels,
            reduced,
            1,
        )

        self.dw3 = nn.Conv1d(
            reduced,
            reduced,
            3,
            padding=1,
            groups=reduced,
        )

        self.dw5 = nn.Conv1d(
            reduced,
            reduced,
            5,
            padding=2,
            groups=reduced,
        )

        self.dw9 = nn.Conv1d(
            reduced,
            reduced,
            9,
            padding=4,
            groups=reduced,
        )

        self.expand = nn.Conv1d(
            reduced * 3,
            channels,
            1,
        )

        self.refine = nn.Sequential(

            nn.Conv1d(
                channels,
                channels,
                1,
            ),

            nn.GELU()

        )

    def forward(self, x):

        identity = x

        x = self.reduce(x)

        x3 = self.dw3(x)

        x5 = self.dw5(x)

        x9 = self.dw9(x)

        x = torch.cat(
            [x3, x5, x9],
            dim=1,
        )

        x = self.expand(x)

        x = self.refine(x)

        return x + identity

# ==========================================================
# Adaptive Gated Feature Fusion (Stable V2)
# ==========================================================

class AdaptiveGatedFeatureFusion(nn.Module):

    def __init__(self, channels=LATENT_CHANNELS):

        super().__init__()

        self.gate = nn.Sequential(

            nn.Conv1d(
                channels,
                channels,
                kernel_size=1,
                bias=False,
            ),

            nn.BatchNorm1d(channels),

            nn.Sigmoid(),

        )

    def forward(

        self,

        original,

        enhanced,

    ):

        gate = self.gate(original)

        fused = original + 0.1 * gate * enhanced

        return fused
        
# ==========================================================
# Speaker Count Head
# ==========================================================

class SpeakerCountHead(nn.Module):

    def __init__(
        self,
        channels=LATENT_CHANNELS,
        hidden_dim=HIDDEN_DIM,
        max_speakers=MAX_SPEAKERS,
    ):

        super().__init__()

        self.pool = nn.AdaptiveAvgPool1d(1)

        self.classifier = nn.Sequential(

            nn.Linear(channels, hidden_dim),

            nn.GELU(),

            nn.Dropout(0.2),

            nn.Linear(hidden_dim, max_speakers)

        )

    def forward(self, x):

        x = self.pool(x)

        x = x.squeeze(-1)

        return self.classifier(x)

# ==========================================================
# Confidence Head
# ==========================================================

class ConfidenceHead(nn.Module):

    def __init__(
        self,
        channels=LATENT_CHANNELS,
        hidden_dim=HIDDEN_DIM,
        max_speakers=MAX_SPEAKERS,
    ):

        super().__init__()

        self.pool = nn.AdaptiveAvgPool1d(1)

        self.classifier = nn.Sequential(

            nn.Linear(channels, hidden_dim),
        
            nn.GELU(),
        
            nn.Dropout(0.2),
        
            nn.Linear(hidden_dim, max_speakers),
        
        )

    def forward(self, x):

        x = self.pool(x)

        x = x.squeeze(-1)

        return self.classifier(x)

# ==========================================================
# Enhanced Adaptive MossFormer
# ==========================================================

class EnhancedAdaptiveMossFormer(nn.Module):

    def __init__(self, config):

        super().__init__()

        self.backbone = Mossformer2Wrapper(config)

        self.backbone.loadPretrained()

        self.encoder = self.backbone.encoder

        self.masknet = self.backbone.masknet

        self.decoder = self.backbone.decoder

        self.num_spks = self.backbone.num_spks

        self.amtm = AdaptiveMultiScaleTemporalModule()

        self.agff = AdaptiveGatedFeatureFusion()

        self.speaker_counter = SpeakerCountHead()

        self.confidence_head = ConfidenceHead()

        for p in self.encoder.parameters():
            p.requires_grad = False

        for p in self.masknet.parameters():
            p.requires_grad = True

        for p in self.decoder.parameters():
            p.requires_grad = False

        # Train AMTM
        for p in self.amtm.parameters():
            p.requires_grad = True
        
        # Train AGFF
        for p in self.agff.parameters():
            p.requires_grad = True
        
        # Train Speaker Counter
        for p in self.speaker_counter.parameters():
            p.requires_grad = True
        
        # Train Confidence Head
        for p in self.confidence_head.parameters():
            p.requires_grad = True

    @property
    def device(self):
        return next(self.parameters()).device

    def forward(self, mixture):

        mix_w = self.encoder(mixture)

        enhanced = self.amtm(mix_w)

        enhanced = torch.tanh(enhanced)
        
        mix_w = self.agff(
            mix_w,
            enhanced,
        )

        speaker_logits = self.speaker_counter(mix_w)

        confidence = self.confidence_head(mix_w)

        est_mask = self.masknet(mix_w)

        mix_stack = torch.stack(
            [mix_w] * self.num_spks
        )

        sep_h = mix_stack * est_mask

        est_source = torch.cat(
            [
                self.decoder(sep_h[i]).unsqueeze(-1)
                for i in range(self.num_spks)
            ],
            dim=-1,
        )

        T_origin = mixture.size(1)

        T_est = est_source.size(1)

        if T_origin > T_est:

            est_source = F.pad(
                est_source,
                (0,0,0,T_origin-T_est)
            )

        else:

            est_source = est_source[:, :T_origin, :]

        return {

            "audio": est_source,

            "speaker_logits": speaker_logits,

            "confidence": confidence

        }

# ==========================================================
# Verify Model
# ==========================================================

gc.collect()

torch.cuda.empty_cache()

model = EnhancedAdaptiveMossFormer(
    mossformer2_wsj0mix_3spk
).to(DEVICE)

model.eval()

with torch.no_grad():

    outputs = model(waveform)

print("="*60)

print(outputs["audio"].shape)

print(outputs["speaker_logits"].shape)

print(outputs["confidence"].shape)

print("="*60)

total = sum(
    p.numel()
    for p in model.parameters()
)

trainable = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Total :", total)

print("Trainable :", trainable)

# ==========================================================
# Loss Function
# ==========================================================

class EnhancedMossFormerLoss(nn.Module):

    def __init__(
        self,
        separation_weight=0.75,
        speaker_weight=0.15,
        confidence_weight=0.10,
    ):

        super().__init__()

        self.log_sigma_sep = nn.Parameter(torch.zeros(1))
        self.log_sigma_spk = nn.Parameter(torch.zeros(1))
        self.log_sigma_conf = nn.Parameter(torch.zeros(1))

        self.separation_loss = nn.L1Loss()

        self.speaker_loss = nn.CrossEntropyLoss()

        self.confidence_loss = nn.BCEWithLogitsLoss()

    def forward(

        self,

        outputs,

        target_audio,

        target_num_speakers,

        target_confidence,

    ):
        print("audio NaN :", torch.isnan(outputs["audio"]).any().item())
        print("target NaN:", torch.isnan(target_audio).any().item())
        
        print("audio Inf :", torch.isinf(outputs["audio"]).any().item())
        print("target Inf:", torch.isinf(target_audio).any().item())
        
        sep = self.separation_loss(
            outputs["audio"],
            target_audio,
        )
        

        spk = self.speaker_loss(

            outputs["speaker_logits"],

            target_num_speakers,

        )

        conf = self.confidence_loss(

            outputs["confidence"],

            target_confidence,

        )

        weight_sep = torch.exp(
            torch.clamp(-self.log_sigma_sep, -5, 5)
        )
        
        weight_spk = torch.exp(
            torch.clamp(-self.log_sigma_spk, -5, 5)
        )
        
        weight_conf = torch.exp(
            torch.clamp(-self.log_sigma_conf, -5, 5)
        )
        
        loss_sep = (
            weight_sep * sep
            + self.log_sigma_sep
        )
        
        loss_spk = (
            weight_spk * spk
            + self.log_sigma_spk
        )
        
        loss_conf = (
            weight_conf * conf
            + self.log_sigma_conf
        )
        
        total = (
            loss_sep
            + loss_spk
            + loss_conf
        )

        print(
            f"log_sigma_sep  : {self.log_sigma_sep.item():.4f}"
        )
        print(
            f"log_sigma_spk  : {self.log_sigma_spk.item():.4f}"
        )
        print(
            f"log_sigma_conf : {self.log_sigma_conf.item():.4f}"
        )

        return {

            "total_loss": total,

            "separation_loss": sep,

            "speaker_loss": spk,

            "confidence_loss": conf,

            "log_sigma_sep": self.log_sigma_sep.detach(),

            "log_sigma_spk": self.log_sigma_spk.detach(),
            
            "log_sigma_conf": self.log_sigma_conf.detach(),

        }

# ==========================================================
# Optimizer
# ==========================================================

criterion = EnhancedMossFormerLoss().to(DEVICE)
print(next(criterion.parameters()).device)

optimizer = optim.AdamW(

    [
        {
            "params": model.parameters(),
        },
        {
            "params": criterion.parameters(),
        },
    ],

    lr=1e-4,

    weight_decay=1e-5,

)

scheduler = optim.lr_scheduler.CosineAnnealingLR(

    optimizer,

    T_max=20,

)

scaler = torch.amp.GradScaler("cuda")

print("Optimizer Ready")

# ==========================================================
# Dataset
# ==========================================================

class MossFormerDataset(Dataset):

    def __init__(

        self,

        root_dir,

    ):

        self.mixtures = sorted(

            glob.glob(

                os.path.join(

                    root_dir,

                    "*_mix.wav",

                )

            )

        )

    def __len__(self):

        return len(self.mixtures)

    def __getitem__(

        self,

        idx,

    ):

        mix_path = self.mixtures[idx]

        mixture, _ = torchaudio.load(

            mix_path

        )

        src1, _ = torchaudio.load(

            mix_path.replace(

                "_mix.wav",

                "_source1.wav",

            )

        )

        src2, _ = torchaudio.load(

            mix_path.replace(

                "_mix.wav",

                "_source2.wav",

            )

        )

        src3, _ = torchaudio.load(

            mix_path.replace(

                "_mix.wav",

                "_source3.wav",

            )

        )

        length = mixture.shape[1]

        if length > CROP_SIZE:

            start = torch.randint(

                0,

                length - CROP_SIZE,

                (1,),

            ).item()

            end = start + CROP_SIZE

            mixture = mixture[:, start:end]

            src1 = src1[:, start:end]

            src2 = src2[:, start:end]

            src3 = src3[:, start:end]

        sources = torch.stack(

            [

                src1.squeeze(0),

                src2.squeeze(0),

                src3.squeeze(0),

            ],

            dim=-1,

        )

        return {

            "mixture": mixture,

            "sources": sources,

            "num_speakers": torch.tensor(

                3,

                dtype=torch.long,

            ),

            "confidence": torch.tensor(

                [

                    1.,

                    1.,

                    1.,

                    0.,

                    0.,

                    0.,

                    0.,

                    0.,

                ],

                dtype=torch.float32,

            ),

        }

# ==========================================================
# DataLoader
# ==========================================================

def collate_fn(batch):

    mixtures = [

        x["mixture"].squeeze()

        for x in batch

    ]

    mixtures = pad_sequence(

        mixtures,

        batch_first=True,

    )

    sources = [

        x["sources"]

        for x in batch

    ]

    max_len = max(

        s.shape[0]

        for s in sources

    )

    padded = []

    for s in sources:

        if s.shape[0] < max_len:

            s = torch.cat(

                [

                    s,

                    torch.zeros(

                        max_len - s.shape[0],

                        3,

                    ),

                ],

                dim=0,

            )

        padded.append(s)

    return {

        "mixture": mixtures,

        "sources": torch.stack(padded),

        "num_speakers": torch.stack(

            [

                x["num_speakers"]

                for x in batch

            ]

        ),

        "confidence": torch.stack(

            [

                x["confidence"]

                for x in batch

            ]

        ),

    }

train_dataset = MossFormerDataset(

    "MossFormer2_samples_WSJ0-3mix"

)

train_loader = DataLoader(

    train_dataset,

    batch_size=1,

    shuffle=True,

    collate_fn=collate_fn,

)

print(len(train_dataset))
print(len(train_loader))

# ==========================================================
# Train One Epoch
# ==========================================================

def train_one_epoch(

    model,

    dataloader,

    optimizer,

    criterion,

    scaler,

    scheduler=None,

):

    model.train()

    running_loss = 0.0
    running_sep = 0.0
    running_spk = 0.0
    running_conf = 0.0

    progress = tqdm(
        dataloader,
        desc="Training",
    )

    for batch in progress:

        mixture = batch["mixture"].to(
            DEVICE,
            non_blocking=True,
        )

        sources = batch["sources"].to(
            DEVICE,
            non_blocking=True,
        )

        num_speakers = batch["num_speakers"].to(
            DEVICE,
            non_blocking=True,
        )

        confidence = batch["confidence"].to(
            DEVICE,
            non_blocking=True,
        )

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(

            device_type="cuda",

            enabled=False #torch.cuda.is_available(),

        ):

            outputs = model(mixture)

            losses = criterion(

                outputs,

                sources,

                num_speakers,

                confidence,

            )

            if torch.isnan(losses["separation_loss"]):
                print("NaN separation loss detected")
                break
            if torch.isnan(outputs["audio"]).any():
                print("NaN in separated audio")
                
            print("\n========== Forward Check ==========")

            for name, tensor in outputs.items():

                print(

                    name,

                    "NaN:", torch.isnan(tensor).any().item(),

                    "Inf:", torch.isinf(tensor).any().item(),

                )

            print("Total Loss :", losses["total_loss"].item())
            print("Sep Loss   :", losses["separation_loss"].item())
            print("Spk Loss   :", losses["speaker_loss"].item())
            print("Conf Loss  :", losses["confidence_loss"].item())

        scaler.scale(
            losses["total_loss"]
        ).backward()

        print("\n========== Gradient Check ==========")

        for name, param in model.named_parameters():

            if param.grad is None:
                continue

            if torch.isnan(param.grad).any():

                print(f"NaN Gradient : {name}")

                break

            if torch.isinf(param.grad).any():

                print(f"Inf Gradient : {name}")

                break

        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(

            model.parameters(),

            max_norm=1.0,

        )

        scaler.step(optimizer)

        scaler.update()

        for name, param in model.named_parameters():

            if torch.isnan(param).any():

                print(f"NaN weights in {name}")

                raise RuntimeError("NaN weights detected")

        running_loss += losses["total_loss"].item()
        running_sep += losses["separation_loss"].item()
        running_spk += losses["speaker_loss"].item()
        running_conf += losses["confidence_loss"].item()

        progress.set_postfix(

            Loss=f"{running_loss/(progress.n+1):.4f}",

            Sep=f"{running_sep/(progress.n+1):.4f}",

            Spk=f"{running_spk/(progress.n+1):.4f}",

            Conf=f"{running_conf/(progress.n+1):.4f}",

        )

    if scheduler is not None:

        scheduler.step()

    return {

        "loss": running_loss / len(dataloader),

        "separation_loss": running_sep / len(dataloader),

        "speaker_loss": running_spk / len(dataloader),

        "confidence_loss": running_conf / len(dataloader),

    }

# ==========================================================
# Validation
# ==========================================================

@torch.no_grad()
def validate(

    model,

    dataloader,

    criterion,

):

    model.eval()

    running = 0.0
    running_sisdr = 0.0

    for batch in dataloader:

        mixture = batch["mixture"].to(DEVICE)

        target_audio = batch["sources"].to(DEVICE)

        target_num_speakers = batch["num_speakers"].to(DEVICE)

        target_confidence = batch["confidence"].to(DEVICE)

        outputs = model(mixture)

        losses = criterion(

            outputs,

            target_audio,

            target_num_speakers,

            target_confidence,

        )

        score = sisdr(

            outputs["audio"].permute(0, 2, 1),

            target_audio.permute(0, 2, 1),

        )

        running += losses["total_loss"].item()

        running_sisdr += score.item()

    return {

        "loss": running / len(dataloader),

        "sisdr": running_sisdr / len(dataloader),

    }

# ==========================================================
# Train
# ==========================================================

gc.collect()

torch.cuda.empty_cache()

best_loss = float("inf")

NUM_EPOCHS = 20

for epoch in range(NUM_EPOCHS):

    print("\n" + "=" * 60)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
    print("=" * 60)

    history = train_one_epoch(

        model=model,

        dataloader=train_loader,

        optimizer=optimizer,

        criterion=criterion,

        scaler=scaler,

        scheduler=scheduler,

    )

    metrics = validate(

        model,

        train_loader,

        criterion,

    )

    print("\nTraining")

    for key, value in history.items():

        print(f"{key:20}: {value:.5f}")

    print("\nValidation")

    print(f"Loss               : {metrics['loss']:.5f}")

    print(f"SI-SDR             : {metrics['sisdr']:.5f}")

    if metrics["loss"] < best_loss:

        best_loss = metrics["loss"]

        torch.save(

            {

                "model": model.state_dict(),

                "optimizer": optimizer.state_dict(),

                "epoch": epoch,

                "loss": metrics["loss"],

            },

            "best_enhanced_mossformer.pth",

        )

        print("\n✓ Best model saved.")

# ==========================================================
# Block 17
# Inference using Best Model + Save WAV Files
# ==========================================================

import os
import torch
import torchaudio
from tqdm import tqdm

SAVE_DIR = "/kaggle/working/separated_audio"

os.makedirs(
    SAVE_DIR,
    exist_ok=True,
)

# ----------------------------------------------------------
# Load Best Checkpoint
# ----------------------------------------------------------

checkpoint = torch.load(
    "best_enhanced_mossformer.pth",
    map_location=DEVICE,
)

model.load_state_dict(checkpoint["model"])

model.to(DEVICE)
model.eval()

print(f"Loaded best model from epoch {checkpoint['epoch'] + 1}")

# ----------------------------------------------------------
# Run inference on all mixtures
# ----------------------------------------------------------

mix_files = train_dataset.mixtures      # Use val_dataset.mixtures if preferred

with torch.no_grad():

    for mix_path in tqdm(mix_files):

        mixture, sr = torchaudio.load(mix_path)

        mixture = mixture.to(DEVICE)

        outputs = model(mixture)

        separated = outputs["audio"].cpu()

        sample_name = os.path.basename(
            mix_path
        ).replace("_mix.wav", "")

        sample_dir = os.path.join(
            SAVE_DIR,
            sample_name,
        )

        os.makedirs(
            sample_dir,
            exist_ok=True,
        )

        # --------------------------------------------------
        # Save mixture
        # --------------------------------------------------

        torchaudio.save(
            os.path.join(
                sample_dir,
                "mixture.wav",
            ),
            mixture.cpu(),
            sr,
        )

        # --------------------------------------------------
        # Save predicted speakers
        # --------------------------------------------------

        for spk in range(separated.shape[-1]):

            torchaudio.save(

                os.path.join(
                    sample_dir,
                    f"pred_speaker_{spk+1}.wav",
                ),

                separated[:, :, spk],

                sr,

            )

        # --------------------------------------------------
        # Save ground-truth speakers (optional)
        # --------------------------------------------------

        for spk in range(1, 4):

            gt_path = mix_path.replace(
                "_mix.wav",
                f"_source{spk}.wav",
            )

            if os.path.exists(gt_path):

                gt_audio, _ = torchaudio.load(gt_path)

                torchaudio.save(

                    os.path.join(
                        sample_dir,
                        f"gt_speaker_{spk}.wav",
                    ),

                    gt_audio,

                    sr,

                )

print("\n All audio files have been saved.")

print(f"\nSaved to:\n{SAVE_DIR}")

mossformer2-wsj0mix-3spk config loaded
model initialised on cuda:0
Waveform : torch.Size([1, 44920])
Encoder  : torch.Size([1, 512, 5614])
mossformer2-wsj0mix-3spk config loaded
model initialised on cuda:0
torch.Size([1, 44920, 3])
torch.Size([1, 8])
torch.Size([1, 8])
Total : 57056002
Trainable : 57039618
cuda:0
Optimizer Ready
38
38

Epoch 1/20


Training:   0%|          | 0/38 [00:00<?, ?it/s]

audio NaN : False
target NaN: False
audio Inf : False
target Inf: False
log_sigma_sep  : 0.0000
log_sigma_spk  : 0.0000
log_sigma_conf : 0.0000

========== Forward Check ==========
audio NaN: False Inf: False
speaker_logits NaN: False Inf: False
confidence NaN: False Inf: False
Total Loss : 3.1681313514709473
Sep Loss   : 0.40434399247169495
Spk Loss   : 2.0659263134002686
Conf Loss  : 0.6978610157966614

========== Gradient Check ==========
audio NaN : False
target NaN: False
audio Inf : False
target Inf: False
log_sigma_sep  : -0.0001
log_sigma_spk  : 0.0001
log_sigma_conf : -0.0001

========== Forward Check ==========
audio NaN: False Inf: False
speaker_logits NaN: False Inf: False
confidence NaN: False Inf: False
Total Loss : 3.204319477081299
Sep Loss   : 0.41943415999412537
Spk Loss   : 2.0890731811523438
Conf Loss  : 0.6960093975067139

========== Gradient Check ==========
audio NaN : False
target NaN: False
audio Inf : False
target Inf: False
log_sigma_sep  : -0.0002
log_sigma_

Training:   0%|          | 0/38 [00:00<?, ?it/s]

audio NaN : False
target NaN: False
audio Inf : False
target Inf: False
log_sigma_sep  : -0.0039
log_sigma_spk  : 0.0037
log_sigma_conf : -0.0039

========== Forward Check ==========
audio NaN: False Inf: False
speaker_logits NaN: False Inf: False
confidence NaN: False Inf: False
Total Loss : 2.4597721099853516
Sep Loss   : 0.13844066858291626
Spk Loss   : 1.7067065238952637
Conf Loss  : 0.6220283508300781

========== Gradient Check ==========
audio NaN : False
target NaN: False
audio Inf : False
target Inf: False
log_sigma_sep  : -0.0040
log_sigma_spk  : 0.0038
log_sigma_conf : -0.0040

========== Forward Check ==========
audio NaN: False Inf: False
speaker_logits NaN: False Inf: False
confidence NaN: False Inf: False
Total Loss : 2.3722376823425293
Sep Loss   : 0.09224247187376022
Spk Loss   : 1.6671106815338135
Conf Loss  : 0.620527982711792

========== Gradient Check ==========
audio NaN : False
target NaN: False
audio Inf : False
target Inf: False
log_sigma_sep  : -0.0041
log_sigm

...

audio NaN : False
target NaN: False
audio Inf : False
target Inf: False
log_sigma_sep  : -0.0420
log_sigma_spk  : -0.0284
log_sigma_conf : -0.0485

========== Forward Check ==========
audio NaN: False Inf: False
speaker_logits NaN: False Inf: False
confidence NaN: False Inf: False
Total Loss : -0.09245029836893082
Sep Loss   : 0.01612952910363674
Spk Loss   : 0.005358143709599972
Conf Loss  : 0.003940167836844921

========== Gradient Check ==========
audio NaN : False
target NaN: False
audio Inf : False
target Inf: False
log_sigma_sep  : -0.0420
log_sigma_spk  : -0.0284
log_sigma_conf : -0.0485

========== Forward Check ==========
audio NaN: False Inf: False
speaker_logits NaN: False Inf: False
confidence NaN: False Inf: False
Total Loss : -0.09547346830368042
Sep Loss   : 0.013046946376562119
Spk Loss   : 0.003946018870919943
Conf Loss  : 0.005514035001397133

========== Gradient Check ==========
audio NaN : False
target NaN: False
audio Inf : False
target Inf: False
log_sigma_sep  : 

Training:   0%|          | 0/38 [00:00<?, ?it/s]

audio NaN : False
target NaN: False
audio Inf : False
target Inf: False
log_sigma_sep  : -0.0421
log_sigma_spk  : -0.0285
log_sigma_conf : -0.0486

========== Forward Check ==========
audio NaN: False Inf: False
speaker_logits NaN: False Inf: False
confidence NaN: False Inf: False
Total Loss : -0.09341511875391006
Sep Loss   : 0.015806114301085472
Spk Loss   : 0.0024441389832645655
Conf Loss  : 0.006472303997725248

========== Gradient Check ==========
audio NaN : False
target NaN: False
audio Inf : False
target Inf: False
log_sigma_sep  : -0.0421
log_sigma_spk  : -0.0285
log_sigma_conf : -0.0486

========== Forward Check ==========
audio NaN: False Inf: False
speaker_logits NaN: False Inf: False
confidence NaN: False Inf: False
Total Loss : -0.09761520475149155
Sep Loss   : 0.015030475333333015
Spk Loss   : 0.0024563875049352646
Conf Loss  : 0.0032318630255758762

========== Gradient Check ==========
audio NaN : False
target NaN: False
audio Inf : False
target Inf: False
log_sigma_sep

100%|██████████| 38/38 [00:32<00:00,  1.16it/s]


 All audio files have been saved.

Saved to:
/kaggle/working/separated_audio


In [5]:
import os

print(os.listdir("/kaggle/input/datasets"))

['garvs777']


In [6]:
import os

for root, dirs, files in os.walk("/kaggle/input/datasets"):
    print(root)
    if root.count("/") > 8:
        break

/kaggle/input/datasets
/kaggle/input/datasets/garvs777
/kaggle/input/datasets/garvs777/libri3mix
/kaggle/input/datasets/garvs777/libri3mix/Libri3Mix
/kaggle/input/datasets/garvs777/libri3mix/Libri3Mix/wav8k
/kaggle/input/datasets/garvs777/libri3mix/Libri3Mix/wav8k/min
/kaggle/input/datasets/garvs777/libri3mix/Libri3Mix/wav8k/min/train-360


In [7]:
!find "/kaggle/input/datasets/garvs777/libri3mix/Libri3Mix/wav8k/min/train-360" -maxdepth 1 -type d


/kaggle/input/datasets/garvs777/libri3mix/Libri3Mix/wav8k/min/train-360
/kaggle/input/datasets/garvs777/libri3mix/Libri3Mix/wav8k/min/train-360/mix_clean
/kaggle/input/datasets/garvs777/libri3mix/Libri3Mix/wav8k/min/train-360/s1
/kaggle/input/datasets/garvs777/libri3mix/Libri3Mix/wav8k/min/train-360/s2
/kaggle/input/datasets/garvs777/libri3mix/Libri3Mix/wav8k/min/train-360/s3


In [8]:
import torchaudio
import os

sample = sorted(os.listdir(
    "/kaggle/input/datasets/garvs777/libri3mix/Libri3Mix/wav8k/min/train-360/mix_clean"
))[0]

waveform, sr = torchaudio.load(
    f"/kaggle/input/datasets/garvs777/libri3mix/Libri3Mix/wav8k/min/train-360/mix_clean/{sample}"
)

print("Sample rate:", sr)
print("Shape:", waveform.shape)


Sample rate: 8000
Shape: torch.Size([1, 53680])


In [ ]:
#Version 2 Libri3Mix-training

# ==========================================================
# Load Pretrained Model
# ==========================================================

backbone = Mossformer2Wrapper(
    mossformer2_wsj0mix_3spk
)

backbone.loadPretrained()

backbone.eval()

waveform, sr = torchaudio.load(
    "MossFormer2_samples_WSJ0-3mix/item0_mix.wav"
)

waveform = waveform.to(backbone.device)

with torch.no_grad():

    mix_w = backbone.encoder(waveform)

print("Waveform :", waveform.shape)
print("Encoder  :", mix_w.shape)

# ==========================================================
# Adaptive Multi Scale Temporal Module
# ==========================================================

class AdaptiveMultiScaleTemporalModule(nn.Module):

    def __init__(self, channels=LATENT_CHANNELS):

        super().__init__()

        reduced = channels // 4

        self.reduce = nn.Conv1d(
            channels,
            reduced,
            1,
        )

        self.dw3 = nn.Conv1d(
            reduced,
            reduced,
            3,
            padding=1,
            groups=reduced,
        )

        self.dw5 = nn.Conv1d(
            reduced,
            reduced,
            5,
            padding=2,
            groups=reduced,
        )

        self.dw9 = nn.Conv1d(
            reduced,
            reduced,
            9,
            padding=4,
            groups=reduced,
        )

        self.expand = nn.Conv1d(
            reduced * 3,
            channels,
            1,
        )

        self.refine = nn.Sequential(

            nn.Conv1d(
                channels,
                channels,
                1,
            ),

            nn.GELU()

        )

    def forward(self, x):

        identity = x

        x = self.reduce(x)

        x3 = self.dw3(x)

        x5 = self.dw5(x)

        x9 = self.dw9(x)

        x = torch.cat(
            [x3, x5, x9],
            dim=1,
        )

        x = self.expand(x)

        x = self.refine(x)

        return x + identity

# ==========================================================
# Adaptive Gated Feature Fusion (Stable V2)
# ==========================================================

class AdaptiveGatedFeatureFusion(nn.Module):

    def __init__(self, channels=LATENT_CHANNELS):

        super().__init__()

        self.gate = nn.Sequential(

            nn.Conv1d(
                channels,
                channels,
                kernel_size=1,
                bias=False,
            ),

            nn.BatchNorm1d(channels),

            nn.Sigmoid(),

        )

    def forward(

        self,

        original,

        enhanced,

    ):

        gate = self.gate(original)

        fused = original + 0.1 * gate * enhanced

        return fused
        
# ==========================================================
# Speaker Count Head
# ==========================================================

class SpeakerCountHead(nn.Module):

    def __init__(
        self,
        channels=LATENT_CHANNELS,
        hidden_dim=HIDDEN_DIM,
        max_speakers=MAX_SPEAKERS,
    ):

        super().__init__()

        self.pool = nn.AdaptiveAvgPool1d(1)

        self.classifier = nn.Sequential(

            nn.Linear(channels, hidden_dim),

            nn.GELU(),

            nn.Dropout(0.2),

            nn.Linear(hidden_dim, max_speakers)

        )

    def forward(self, x):

        x = self.pool(x)

        x = x.squeeze(-1)

        return self.classifier(x)

# ==========================================================
# Confidence Head
# ==========================================================

class ConfidenceHead(nn.Module):

    def __init__(
        self,
        channels=LATENT_CHANNELS,
        hidden_dim=HIDDEN_DIM,
        max_speakers=MAX_SPEAKERS,
    ):

        super().__init__()

        self.pool = nn.AdaptiveAvgPool1d(1)

        self.classifier = nn.Sequential(

            nn.Linear(channels, hidden_dim),
        
            nn.GELU(),
        
            nn.Dropout(0.2),
        
            nn.Linear(hidden_dim, max_speakers),
        
        )

    def forward(self, x):

        x = self.pool(x)

        x = x.squeeze(-1)

        return self.classifier(x)

# ==========================================================
# Enhanced Adaptive MossFormer
# ==========================================================

class EnhancedAdaptiveMossFormer(nn.Module):

    def __init__(self, config):

        super().__init__()

        self.backbone = Mossformer2Wrapper(config)

        self.backbone.loadPretrained()

        self.encoder = self.backbone.encoder

        self.masknet = self.backbone.masknet

        self.decoder = self.backbone.decoder

        self.num_spks = self.backbone.num_spks

        self.amtm = AdaptiveMultiScaleTemporalModule()

        self.agff = AdaptiveGatedFeatureFusion()

        self.speaker_counter = SpeakerCountHead()

        self.confidence_head = ConfidenceHead()

        for p in self.encoder.parameters():
            p.requires_grad = False

        for p in self.masknet.parameters():
            p.requires_grad = True

        for p in self.decoder.parameters():
            p.requires_grad = False

        # Train AMTM
        for p in self.amtm.parameters():
            p.requires_grad = True
        
        # Train AGFF
        for p in self.agff.parameters():
            p.requires_grad = True
        
        # Train Speaker Counter
        for p in self.speaker_counter.parameters():
            p.requires_grad = True
        
        # Train Confidence Head
        for p in self.confidence_head.parameters():
            p.requires_grad = True

    @property
    def device(self):
        return next(self.parameters()).device

    def forward(self, mixture):

        mix_w = self.encoder(mixture)

        enhanced = self.amtm(mix_w)

        enhanced = torch.tanh(enhanced)
        
        mix_w = self.agff(
            mix_w,
            enhanced,
        )

        speaker_logits = self.speaker_counter(mix_w)

        confidence = self.confidence_head(mix_w)

        est_mask = self.masknet(mix_w)

        mix_stack = torch.stack(
            [mix_w] * self.num_spks
        )

        sep_h = mix_stack * est_mask

        est_source = torch.cat(
            [
                self.decoder(sep_h[i]).unsqueeze(-1)
                for i in range(self.num_spks)
            ],
            dim=-1,
        )

        T_origin = mixture.size(1)

        T_est = est_source.size(1)

        if T_origin > T_est:

            est_source = F.pad(
                est_source,
                (0,0,0,T_origin-T_est)
            )

        else:

            est_source = est_source[:, :T_origin, :]

        return {

            "audio": est_source,

            "speaker_logits": speaker_logits,

            "confidence": confidence

        }

# ==========================================================
# Verify Model
# ==========================================================

gc.collect()

torch.cuda.empty_cache()

model = EnhancedAdaptiveMossFormer(
    mossformer2_wsj0mix_3spk
).to(DEVICE)

model.eval()

with torch.no_grad():

    outputs = model(waveform)

print("="*60)

print(outputs["audio"].shape)

print(outputs["speaker_logits"].shape)

print(outputs["confidence"].shape)

print("="*60)

total = sum(
    p.numel()
    for p in model.parameters()
)

trainable = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Total :", total)

print("Trainable :", trainable)

# ==========================================================
# Loss Function
# ==========================================================

class EnhancedMossFormerLoss(nn.Module):

    def __init__(
        self,
        separation_weight=0.75,
        speaker_weight=0.15,
        confidence_weight=0.10,
    ):

        super().__init__()

        self.log_sigma_sep = nn.Parameter(torch.zeros(1))
        self.log_sigma_spk = nn.Parameter(torch.zeros(1))
        self.log_sigma_conf = nn.Parameter(torch.zeros(1))

        self.separation_loss = nn.L1Loss()

        self.speaker_loss = nn.CrossEntropyLoss()

        self.confidence_loss = nn.BCEWithLogitsLoss()

    def forward(

        self,

        outputs,

        target_audio,

        target_num_speakers,

        target_confidence,

    ):
        print("audio NaN :", torch.isnan(outputs["audio"]).any().item())
        print("target NaN:", torch.isnan(target_audio).any().item())
        
        print("audio Inf :", torch.isinf(outputs["audio"]).any().item())
        print("target Inf:", torch.isinf(target_audio).any().item())
        
        sep = self.separation_loss(
            outputs["audio"],
            target_audio,
        )
        

        spk = self.speaker_loss(

            outputs["speaker_logits"],

            target_num_speakers,

        )

        conf = self.confidence_loss(

            outputs["confidence"],

            target_confidence,

        )

        weight_sep = torch.exp(
            torch.clamp(-self.log_sigma_sep, -5, 5)
        )
        
        weight_spk = torch.exp(
            torch.clamp(-self.log_sigma_spk, -5, 5)
        )
        
        weight_conf = torch.exp(
            torch.clamp(-self.log_sigma_conf, -5, 5)
        )
        
        loss_sep = (
            weight_sep * sep
            + self.log_sigma_sep
        )
        
        loss_spk = (
            weight_spk * spk
            + self.log_sigma_spk
        )
        
        loss_conf = (
            weight_conf * conf
            + self.log_sigma_conf
        )
        
        total = (
            loss_sep
            + loss_spk
            + loss_conf
        )

        print(
            f"log_sigma_sep  : {self.log_sigma_sep.item():.4f}"
        )
        print(
            f"log_sigma_spk  : {self.log_sigma_spk.item():.4f}"
        )
        print(
            f"log_sigma_conf : {self.log_sigma_conf.item():.4f}"
        )

        return {

            "total_loss": total,

            "separation_loss": sep,

            "speaker_loss": spk,

            "confidence_loss": conf,

            "log_sigma_sep": self.log_sigma_sep.detach(),

            "log_sigma_spk": self.log_sigma_spk.detach(),
            
            "log_sigma_conf": self.log_sigma_conf.detach(),

        }

# ==========================================================
# Optimizer
# ==========================================================

criterion = EnhancedMossFormerLoss().to(DEVICE)
print(next(criterion.parameters()).device)

optimizer = optim.AdamW(

    [
        {
            "params": model.parameters(),
        },
        {
            "params": criterion.parameters(),
        },
    ],

    lr=1e-4,

    weight_decay=1e-5,

)

scheduler = optim.lr_scheduler.CosineAnnealingLR(

    optimizer,

    T_max=20,

)

scaler = torch.amp.GradScaler("cuda")

print("Optimizer Ready")

# ==========================================================
# Dataset - Libri3Mix
# ==========================================================

class MossFormerDataset(Dataset):

    def __init__(self, root_dir):

        self.root_dir = root_dir

        self.mix_dir = os.path.join(root_dir, "mix_clean")
        self.s1_dir = os.path.join(root_dir, "s1")
        self.s2_dir = os.path.join(root_dir, "s2")
        self.s3_dir = os.path.join(root_dir, "s3")

        self.mixtures = sorted(
            glob.glob(
                os.path.join(
                    self.mix_dir,
                    "*.wav"
                )
            )
        )
        
        # -------------------------------------------------
        # TEMPORARY: Train on a subset for faster epochs
        # -------------------------------------------------
        MAX_SAMPLES = 3000        # Change to 10000 or 20000 later
        
        self.mixtures = self.mixtures[:MAX_SAMPLES]
        
        print(f"Loaded {len(self.mixtures)} mixtures.")

    def __len__(self):
        return len(self.mixtures)

    def __getitem__(self, idx):

        mix_path = self.mixtures[idx]

        filename = os.path.basename(mix_path)

        s1_path = os.path.join(self.s1_dir, filename)
        s2_path = os.path.join(self.s2_dir, filename)
        s3_path = os.path.join(self.s3_dir, filename)

        mixture, sr = torchaudio.load(mix_path)

        src1, _ = torchaudio.load(s1_path)
        src2, _ = torchaudio.load(s2_path)
        src3, _ = torchaudio.load(s3_path)

        length = mixture.shape[1]

        if length > CROP_SIZE:

            start = torch.randint(
                0,
                length - CROP_SIZE,
                (1,)
            ).item()

            end = start + CROP_SIZE

            mixture = mixture[:, start:end]
            src1 = src1[:, start:end]
            src2 = src2[:, start:end]
            src3 = src3[:, start:end]

        sources = torch.stack(

            [

                src1.squeeze(0),

                src2.squeeze(0),

                src3.squeeze(0),

            ],

            dim=-1,

        )

        return {

            "mixture": mixture,

            "sources": sources,

            "num_speakers": torch.tensor(
                3,
                dtype=torch.long,
            ),

            "confidence": torch.tensor(

                [

                    1.,
                    1.,
                    1.,
                    0.,
                    0.,
                    0.,
                    0.,
                    0.,

                ],

                dtype=torch.float32,

            ),

        }

# ==========================================================
# DataLoader
# ==========================================================

def collate_fn(batch):

    mixtures = [

        x["mixture"].squeeze()

        for x in batch

    ]

    mixtures = pad_sequence(

        mixtures,

        batch_first=True,

    )

    sources = [

        x["sources"]

        for x in batch

    ]

    max_len = max(

        s.shape[0]

        for s in sources

    )

    padded = []

    for s in sources:

        if s.shape[0] < max_len:

            s = torch.cat(

                [

                    s,

                    torch.zeros(

                        max_len - s.shape[0],

                        3,

                    ),

                ],

                dim=0,

            )

        padded.append(s)

    return {

        "mixture": mixtures,

        "sources": torch.stack(padded),

        "num_speakers": torch.stack(

            [

                x["num_speakers"]

                for x in batch

            ]

        ),

        "confidence": torch.stack(

            [

                x["confidence"]

                for x in batch

            ]

        ),

    }

TRAIN_DIR = "/kaggle/input/datasets/garvs777/libri3mix/Libri3Mix/wav8k/min/train-360"

train_dataset = MossFormerDataset(TRAIN_DIR)

train_loader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,
)

print(len(train_dataset))
print(len(train_loader))

# ==========================================================
# Train One Epoch
# ==========================================================

def train_one_epoch(

    model,

    dataloader,

    optimizer,

    criterion,

    scaler,

    scheduler=None,

):

    model.train()

    running_loss = 0.0
    running_sep = 0.0
    running_spk = 0.0
    running_conf = 0.0

    progress = tqdm(
        dataloader,
        desc="Training",
    )

    for batch in progress:

        mixture = batch["mixture"].to(
            DEVICE,
            non_blocking=True,
        )

        sources = batch["sources"].to(
            DEVICE,
            non_blocking=True,
        )

        num_speakers = batch["num_speakers"].to(
            DEVICE,
            non_blocking=True,
        )

        confidence = batch["confidence"].to(
            DEVICE,
            non_blocking=True,
        )

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(

            device_type="cuda",

            enabled=False #torch.cuda.is_available(),

        ):

            outputs = model(mixture)

            losses = criterion(

                outputs,

                sources,

                num_speakers,

                confidence,

            )

            if torch.isnan(losses["separation_loss"]):
                print("NaN separation loss detected")
                break
            if torch.isnan(outputs["audio"]).any():
                print("NaN in separated audio")
                
            print("\n========== Forward Check ==========")

            for name, tensor in outputs.items():

                print(

                    name,

                    "NaN:", torch.isnan(tensor).any().item(),

                    "Inf:", torch.isinf(tensor).any().item(),

                )

            print("Total Loss :", losses["total_loss"].item())
            print("Sep Loss   :", losses["separation_loss"].item())
            print("Spk Loss   :", losses["speaker_loss"].item())
            print("Conf Loss  :", losses["confidence_loss"].item())

        scaler.scale(
            losses["total_loss"]
        ).backward()

        print("\n========== Gradient Check ==========")

        for name, param in model.named_parameters():

            if param.grad is None:
                continue

            if torch.isnan(param.grad).any():

                print(f"NaN Gradient : {name}")

                break

            if torch.isinf(param.grad).any():

                print(f"Inf Gradient : {name}")

                break

        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(

            model.parameters(),

            max_norm=1.0,

        )

        scaler.step(optimizer)

        scaler.update()

        for name, param in model.named_parameters():

            if torch.isnan(param).any():

                print(f"NaN weights in {name}")

                raise RuntimeError("NaN weights detected")

        running_loss += losses["total_loss"].item()
        running_sep += losses["separation_loss"].item()
        running_spk += losses["speaker_loss"].item()
        running_conf += losses["confidence_loss"].item()

        progress.set_postfix(

            Loss=f"{running_loss/(progress.n+1):.4f}",

            Sep=f"{running_sep/(progress.n+1):.4f}",

            Spk=f"{running_spk/(progress.n+1):.4f}",

            Conf=f"{running_conf/(progress.n+1):.4f}",

        )

    if scheduler is not None:

        scheduler.step()

    return {

        "loss": running_loss / len(dataloader),

        "separation_loss": running_sep / len(dataloader),

        "speaker_loss": running_spk / len(dataloader),

        "confidence_loss": running_conf / len(dataloader),

    }

# ==========================================================
# Validation
# ==========================================================

@torch.no_grad()
def validate(

    model,

    dataloader,

    criterion,

):

    model.eval()

    running = 0.0
    running_sisdr = 0.0

    for batch in dataloader:

        mixture = batch["mixture"].to(DEVICE)

        target_audio = batch["sources"].to(DEVICE)

        target_num_speakers = batch["num_speakers"].to(DEVICE)

        target_confidence = batch["confidence"].to(DEVICE)

        outputs = model(mixture)

        losses = criterion(

            outputs,

            target_audio,

            target_num_speakers,

            target_confidence,

        )

        score = sisdr(

            outputs["audio"].permute(0, 2, 1),

            target_audio.permute(0, 2, 1),

        )

        running += losses["total_loss"].item()

        running_sisdr += score.item()

    return {

        "loss": running / len(dataloader),

        "sisdr": running_sisdr / len(dataloader),

    }

# ==========================================================
# Train
# ==========================================================

gc.collect()

torch.cuda.empty_cache()

best_loss = float("inf")

NUM_EPOCHS = 5

for epoch in range(NUM_EPOCHS):

    print("\n" + "=" * 60)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
    print("=" * 60)

    history = train_one_epoch(

        model=model,

        dataloader=train_loader,

        optimizer=optimizer,

        criterion=criterion,

        scaler=scaler,

        scheduler=scheduler,

    )

    metrics = validate(

        model,

        train_loader,

        criterion,

    )

    print("\nTraining")

    for key, value in history.items():

        print(f"{key:20}: {value:.5f}")

    print("\nValidation")

    print(f"Loss               : {metrics['loss']:.5f}")

    print(f"SI-SDR             : {metrics['sisdr']:.5f}")

    if metrics["loss"] < best_loss:

        best_loss = metrics["loss"]

        torch.save(

            {

                "model": model.state_dict(),

                "optimizer": optimizer.state_dict(),

                "epoch": epoch,

                "loss": metrics["loss"],

            },

            "best_enhanced_mossformer.pth",

        )

        print("\n✓ Best model saved.")

# ==========================================================
# Block 17
# Inference using Best Model + Save WAV Files
# ==========================================================

import os
import torch
import torchaudio
from tqdm import tqdm

SAVE_DIR = "/kaggle/working/separated_audio"

os.makedirs(
    SAVE_DIR,
    exist_ok=True,
)

# ----------------------------------------------------------
# Load Best Checkpoint
# ----------------------------------------------------------

checkpoint = torch.load(
    "best_enhanced_mossformer.pth",
    map_location=DEVICE,
)

model.load_state_dict(checkpoint["model"])

model.to(DEVICE)
model.eval()

print(f"Loaded best model from epoch {checkpoint['epoch'] + 1}")

# ----------------------------------------------------------
# Run inference on all mixtures
# ----------------------------------------------------------

mix_files = train_dataset.mixtures      # Use val_dataset.mixtures if preferred

with torch.no_grad():

    for mix_path in tqdm(mix_files):

        mixture, sr = torchaudio.load(mix_path)

        mixture = mixture.to(DEVICE)

        outputs = model(mixture)

        separated = outputs["audio"].cpu()

        sample_name = os.path.basename(
            mix_path
        ).replace("_mix.wav", "")

        sample_dir = os.path.join(
            SAVE_DIR,
            sample_name,
        )

        os.makedirs(
            sample_dir,
            exist_ok=True,
        )

        # --------------------------------------------------
        # Save mixture
        # --------------------------------------------------

        torchaudio.save(
            os.path.join(
                sample_dir,
                "mixture.wav",
            ),
            mixture.cpu(),
            sr,
        )

        # --------------------------------------------------
        # Save predicted speakers
        # --------------------------------------------------

        for spk in range(separated.shape[-1]):

            torchaudio.save(

                os.path.join(
                    sample_dir,
                    f"pred_speaker_{spk+1}.wav",
                ),

                separated[:, :, spk],

                sr,

            )

        # --------------------------------------------------
        # Save ground-truth speakers (optional)
        # --------------------------------------------------

        for spk in range(1, 4):

            gt_path = mix_path.replace(
                "_mix.wav",
                f"_source{spk}.wav",
            )

            if os.path.exists(gt_path):

                gt_audio, _ = torchaudio.load(gt_path)

                torchaudio.save(

                    os.path.join(
                        sample_dir,
                        f"gt_speaker_{spk}.wav",
                    ),

                    gt_audio,

                    sr,

                )

print("\n All audio files have been saved.")

print(f"\nSaved to:\n{SAVE_DIR}")

mossformer2-wsj0mix-3spk config loaded
model initialised on cuda:0
Waveform : torch.Size([1, 44920])
Encoder  : torch.Size([1, 512, 5614])
mossformer2-wsj0mix-3spk config loaded
model initialised on cuda:0
torch.Size([1, 44920, 3])
torch.Size([1, 8])
torch.Size([1, 8])
Total : 57056002
Trainable : 57039618
cuda:0
Optimizer Ready
Loaded 4500 mixtures.
4500
4500

Epoch 1/5


Training:   0%|          | 0/4500 [00:00<?, ?it/s]

audio NaN : False
target NaN: False
audio Inf : False
target Inf: False
log_sigma_sep  : 0.0000
log_sigma_spk  : 0.0000
log_sigma_conf : 0.0000

========== Forward Check ==========
audio NaN: False Inf: False
speaker_logits NaN: False Inf: False
confidence NaN: False Inf: False
Total Loss : 3.313664436340332
Sep Loss   : 0.5608637928962708
Spk Loss   : 2.049842596054077
Conf Loss  : 0.7029581069946289

========== Gradient Check ==========
audio NaN : False
target NaN: False
audio Inf : False
target Inf: False
log_sigma_sep  : -0.0001
log_sigma_spk  : 0.0001
log_sigma_conf : -0.0001

========== Forward Check ==========
audio NaN: False Inf: False
speaker_logits NaN: False Inf: False
confidence NaN: False Inf: False
Total Loss : 3.095363140106201
Sep Loss   : 0.36569929122924805
Spk Loss   : 2.034958600997925
Conf Loss  : 0.6949027180671692

========== Gradient Check ==========
audio NaN : False
target NaN: False
audio Inf : False
target Inf: False
log_sigma_sep  : -0.0002
log_sigma_spk 

In [ ]:
# Test Single Audio File
# ==========================================================

import os
import torchaudio

# ----------------------------------------------------------
# CHANGE THESE TWO PATHS
# ----------------------------------------------------------

AUDIO_PATH = "/kaggle/input/your_dataset/test.wav"

CHECKPOINT = "/kaggle/working/best_enhanced_mossformer.pth"

SAVE_DIR = "/kaggle/working/test_output"

os.makedirs(SAVE_DIR, exist_ok=True)

# ----------------------------------------------------------
# Load model
# ----------------------------------------------------------

model = EnhancedAdaptiveMossFormer(backbone).to(device)

checkpoint = torch.load(CHECKPOINT, map_location=device)

model.load_state_dict(checkpoint)

model.eval()

# ----------------------------------------------------------
# Load audio
# ----------------------------------------------------------

mixture, sr = torchaudio.load(AUDIO_PATH)

if mixture.shape[0] > 1:
    mixture = mixture.mean(dim=0, keepdim=True)

mixture = mixture.to(device)

# ----------------------------------------------------------
# Inference
# ----------------------------------------------------------

with torch.no_grad():

    separated, speaker_logits, confidence_logits = model(mixture)

# ----------------------------------------------------------
# Predictions
# ----------------------------------------------------------

predicted_speakers = speaker_logits.argmax(dim=-1).item() + 1

confidence = torch.sigmoid(confidence_logits).squeeze().cpu()

print("="*50)
print("Predicted Speakers :", predicted_speakers)
print("Confidence Vector  :", confidence.numpy())
print("="*50)

# ----------------------------------------------------------
# Save separated audio
# ----------------------------------------------------------

separated = separated.squeeze(0).cpu()

for i in range(separated.shape[-1]):

    torchaudio.save(

        os.path.join(

            SAVE_DIR,

            f"speaker_{i+1}.wav"

        ),

        separated[:, i].unsqueeze(0),

        sr,

    )

print(f"\nSeparated audio saved to:\n{SAVE_DIR}")